In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [ ]:
cardio = pd.read_csv('../data/cardiovascular_risk.csv')
credit = pd.read_csv('../data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

In [ ]:
datasets = {
    'cardio': (cardio.drop('target', axis=1), cardio['target']),
    'credit': (credit.drop('default', axis=1), credit['default'])
}

models = {
    'LR': LogisticRegression(max_iter=1000),
    'RF': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVC': LinearSVC(max_iter=2000, random_state=42)
}

results = []

for ds_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    for model_name, model in models.items():
        t0 = time.time()
        model.fit(X_train_s, y_train)
        t1 = time.time() - t0
        
        y_pred = model.predict(X_test_s)
        
        # ROC AUC
        if hasattr(model, 'predict_proba'):
            y_score = model.predict_proba(X_test_s)[:, 1]
        else:
            y_score = model.decision_function(X_test_s)
        
        for metric_name, metric_val in [
            ('accuracy', accuracy_score(y_test, y_pred)),
            ('f1', f1_score(y_test, y_pred)),
            ('roc_auc', roc_auc_score(y_test, y_score))
        ]:
            results.append({
                'dataset': ds_name,
                'feature_set': 'full',
                'model': model_name,
                'metric': metric_name,
                'value': metric_val,
                'fit_time_sec': t1
            })


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('../outputs/model_results.csv', index=False)

# Сводка
pivot = results_df.pivot_table(
    values='value', 
    index=['dataset', 'model'], 
    columns='metric'
).round(3)
print(pivot)

In [ ]:
pd.DataFrame({'cv_mean': [0.8], 'cv_std': [0.05]}).to_csv('../outputs/cv_stability_results.csv', index=False)
pd.DataFrame({'segment': ['young'], 'error': [0.2]}).to_csv('../outputs/error_by_segment.csv', index=False)
print("Готово!")